In [ ]:
#get the parent folder on which this notebook is located
import os
import sys

parent_folder = os.getcwd().replace("\\notebooks","")

#add the parent folder to the system path   
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

parent_folder = os.getcwd().replace("\\notebooks","\\doc-proc-lib")

#add the parent folder to the system path   
if parent_folder not in sys.path:
    sys.path.append(parent_folder)

sys.path.append('C:\github\cmf\doc-proc-solution-accelerator\doc-proc-lib')

In [ ]:
os.environ["APP_CONFIGURATION_URI"] = "https://appcs-REPLACE_ME.azconfig.io"

#from dotenv import load_dotenv
from connectors import CosmosDBClient
from configuration import Configuration

config = Configuration()
cosmos_client = CosmosDBClient(config)

#load_dotenv()

In [ ]:
async def save_prompt(name:str, content:str, container:str="prompts"):
    #content = content.replace("\n", "\\n")
    item = {
        "id": name,
        "system_prompt": content
    }

    await cosmos_client.upsert_document(container, item)

def decode_prompt(name:str, container:str="prompts"):
    item = cosmos_client.get_document(container, name)
    if item is None:
        return None
    prompt = item.get("system_prompt", None)
    if prompt is None:
        return None
    
    prompt = prompt.replace("\\n", "\n")

    return prompt

In [ ]:
graphrag_extration = decode_prompt('graphrag_extration')
print(graphrag_extration)

In [ ]:
graphrag_extration = """
-Goal-
Given a text document that is potentially relevant to this activity and a list of entity types, identify all entities of those types from the text and all relationships among the identified entities.
 
-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types: [{entity_types}]
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"{tuple_delimiter}<entity_name>{tuple_delimiter}<entity_type>{tuple_delimiter}<entity_description>)
 
2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other
- relationship_strength: a numeric score indicating strength of the relationship between the source entity and target entity
 Format each relationship as ("relationship"{tuple_delimiter}<source_entity>{tuple_delimiter}<target_entity>{tuple_delimiter}<relationship_description>{tuple_delimiter}<relationship_strength>)
 
3. Return output in English as a single list of all the entities and relationships identified in steps 1 and 2. Use **{record_delimiter}** as the list delimiter.
 
4. When finished, output {completion_delimiter}
 
######################
-Examples-
######################
Example 1:
Entity_types: ORGANIZATION,PERSON
Text:
The Verdantis's Central Institution is scheduled to meet on Monday and Thursday, with the institution planning to release its latest policy decision on Thursday at 1:30 p.m. PDT, followed by a press conference where Central Institution Chair Martin Smith will take questions. Investors expect the Market Strategy Committee to hold its benchmark interest rate steady in a range of 3.5%-3.75%.
######################
Output:
("entity"{tuple_delimiter}CENTRAL INSTITUTION{tuple_delimiter}ORGANIZATION{tuple_delimiter}The Central Institution is the Federal Reserve of Verdantis, which is setting interest rates on Monday and Thursday)
{record_delimiter}
("entity"{tuple_delimiter}MARTIN SMITH{tuple_delimiter}PERSON{tuple_delimiter}Martin Smith is the chair of the Central Institution)
{record_delimiter}
("entity"{tuple_delimiter}MARKET STRATEGY COMMITTEE{tuple_delimiter}ORGANIZATION{tuple_delimiter}The Central Institution committee makes key decisions about interest rates and the growth of Verdantis's money supply)
{record_delimiter}
("relationship"{tuple_delimiter}MARTIN SMITH{tuple_delimiter}CENTRAL INSTITUTION{tuple_delimiter}Martin Smith is the Chair of the Central Institution and will answer questions at a press conference{tuple_delimiter}9)
{completion_delimiter}

######################
Example 2:
Entity_types: ORGANIZATION
Text:
TechGlobal's (TG) stock skyrocketed in its opening day on the Global Exchange Thursday. But IPO experts warn that the semiconductor corporation's debut on the public markets isn't indicative of how other newly listed companies may perform.

TechGlobal, a formerly public company, was taken private by Vision Holdings in 2014. The well-established chip designer says it powers 85% of premium smartphones.
######################
Output:
("entity"{tuple_delimiter}TECHGLOBAL{tuple_delimiter}ORGANIZATION{tuple_delimiter}TechGlobal is a stock now listed on the Global Exchange which powers 85% of premium smartphones)
{record_delimiter}
("entity"{tuple_delimiter}VISION HOLDINGS{tuple_delimiter}ORGANIZATION{tuple_delimiter}Vision Holdings is a firm that previously owned TechGlobal)
{record_delimiter}
("relationship"{tuple_delimiter}TECHGLOBAL{tuple_delimiter}VISION HOLDINGS{tuple_delimiter}Vision Holdings formerly owned TechGlobal from 2014 until present{tuple_delimiter}5)
{completion_delimiter}

######################
Example 3:
Entity_types: ORGANIZATION,GEO,PERSON
Text:
Five Aurelians jailed for 8 years in Firuzabad and widely regarded as hostages are on their way home to Aurelia.

The swap orchestrated by Quintara was finalized when $8bn of Firuzi funds were transferred to financial institutions in Krohaara, the capital of Quintara.

The exchange initiated in Firuzabad's capital, Tiruzia, led to the four men and one woman, who are also Firuzi nationals, boarding a chartered flight to Krohaara.

They were welcomed by senior Aurelian officials and are now on their way to Aurelia's capital, Cashion.

The Aurelians include 39-year-old businessman Samuel Namara, who has been held in Tiruzia's Alhamia Prison, as well as journalist Durke Bataglani, 59, and environmentalist Meggie Tazbah, 53, who also holds Bratinas nationality.
######################
Output:
("entity"{tuple_delimiter}FIRUZABAD{tuple_delimiter}GEO{tuple_delimiter}Firuzabad held Aurelians as hostages)
{record_delimiter}
("entity"{tuple_delimiter}AURELIA{tuple_delimiter}GEO{tuple_delimiter}Country seeking to release hostages)
{record_delimiter}
("entity"{tuple_delimiter}QUINTARA{tuple_delimiter}GEO{tuple_delimiter}Country that negotiated a swap of money in exchange for hostages)
{record_delimiter}
{record_delimiter}
("entity"{tuple_delimiter}TIRUZIA{tuple_delimiter}GEO{tuple_delimiter}Capital of Firuzabad where the Aurelians were being held)
{record_delimiter}
("entity"{tuple_delimiter}KROHAARA{tuple_delimiter}GEO{tuple_delimiter}Capital city in Quintara)
{record_delimiter}
("entity"{tuple_delimiter}CASHION{tuple_delimiter}GEO{tuple_delimiter}Capital city in Aurelia)
{record_delimiter}
("entity"{tuple_delimiter}SAMUEL NAMARA{tuple_delimiter}PERSON{tuple_delimiter}Aurelian who spent time in Tiruzia's Alhamia Prison)
{record_delimiter}
("entity"{tuple_delimiter}ALHAMIA PRISON{tuple_delimiter}GEO{tuple_delimiter}Prison in Tiruzia)
{record_delimiter}
("entity"{tuple_delimiter}DURKE BATAGLANI{tuple_delimiter}PERSON{tuple_delimiter}Aurelian journalist who was held hostage)
{record_delimiter}
("entity"{tuple_delimiter}MEGGIE TAZBAH{tuple_delimiter}PERSON{tuple_delimiter}Bratinas national and environmentalist who was held hostage)
{record_delimiter}
("relationship"{tuple_delimiter}FIRUZABAD{tuple_delimiter}AURELIA{tuple_delimiter}Firuzabad negotiated a hostage exchange with Aurelia{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}QUINTARA{tuple_delimiter}AURELIA{tuple_delimiter}Quintara brokered the hostage exchange between Firuzabad and Aurelia{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}QUINTARA{tuple_delimiter}FIRUZABAD{tuple_delimiter}Quintara brokered the hostage exchange between Firuzabad and Aurelia{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}SAMUEL NAMARA{tuple_delimiter}ALHAMIA PRISON{tuple_delimiter}Samuel Namara was a prisoner at Alhamia prison{tuple_delimiter}8)
{record_delimiter}
("relationship"{tuple_delimiter}SAMUEL NAMARA{tuple_delimiter}MEGGIE TAZBAH{tuple_delimiter}Samuel Namara and Meggie Tazbah were exchanged in the same hostage release{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}SAMUEL NAMARA{tuple_delimiter}DURKE BATAGLANI{tuple_delimiter}Samuel Namara and Durke Bataglani were exchanged in the same hostage release{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}MEGGIE TAZBAH{tuple_delimiter}DURKE BATAGLANI{tuple_delimiter}Meggie Tazbah and Durke Bataglani were exchanged in the same hostage release{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}SAMUEL NAMARA{tuple_delimiter}FIRUZABAD{tuple_delimiter}Samuel Namara was a hostage in Firuzabad{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}MEGGIE TAZBAH{tuple_delimiter}FIRUZABAD{tuple_delimiter}Meggie Tazbah was a hostage in Firuzabad{tuple_delimiter}2)
{record_delimiter}
("relationship"{tuple_delimiter}DURKE BATAGLANI{tuple_delimiter}FIRUZABAD{tuple_delimiter}Durke Bataglani was a hostage in Firuzabad{tuple_delimiter}2)
{completion_delimiter}

######################
-Real Data-
######################
Entity_types: {entity_types}
Text: {input_text}
######################
Output:
"""

await save_prompt("graphrag_extration", graphrag_extration)

In [ ]:
graphrag_continue = decode_prompt('graphrag_continue')
print(graphrag_continue)

In [ ]:
graphrag_continue = """
MANY entities and relationships were missed in the last extraction. Remember to ONLY emit entities that match any of the previously extracted types. Add them below using the same format:\n
"""

await save_prompt("graphrag_continue", graphrag_continue)

In [ ]:
graphrag_loop = decode_prompt('graphrag_loop')
print(graphrag_loop)

In [ ]:
graphrag_loop = """
It appears some entities and relationships may have still been missed. Answer Y if there are still entities or relationships that need to be added, or N if there are none. Please answer with a single letter Y or N.\n
"""

await save_prompt("graphrag_loop", graphrag_loop)

In [ ]:
pipeline_config = decode_prompt('pipeline_config', container="docproc")
print(pipeline_config)

In [ ]:
pipeline_config = """
# pipeline_config.yaml - Configuration of Pipelines that uses external catalogs


########################################################
# SERVICE INSTANCES
# Service instances - references to services catalog with custom settings
service_instances:

  - name: primary_blob_storage
    service_catalog_id: azure_storage_01 # Reference to service in catalog
    settings:
      account_name: ${STORAGE_ACCOUNT_NAME}
      credential_type: ${STORAGE_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      credential_key: ${STORAGE_ACCOUNT_KEY}

  - name: primary_ai_inference_service
    service_catalog_id: azure_ai_inference_service_01
    settings:
      endpoint: ${AI_FOUNDRY_ACCOUNT_ENDPOINT}
      credential_type: ${AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      api_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}

  - name: primary_ai_embedding_service
    service_catalog_id: azure_ai_embedding_service_01
    settings:
      endpoint: ${AI_FOUNDRY_ACCOUNT_ENDPOINT}
      credential_type: ${AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      api_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}

  - name: primary_document_intelligence_service
    service_catalog_id: azure_document_intelligence_service_01
    settings:
      endpoint: ${AI_FOUNDRY_ACCOUNT_ENDPOINT}
      credential_type: ${AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      api_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}

  - name: primary_ai_search_service
    service_catalog_id: azure_ai_search_service_01
    settings:
      account_name: ${SEARCH_SERVICE_NAME}
      credential_type: ${SEARCH_SERVICE_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      api_key: ${SEARCH_SERVICE_APIKEY}
      api_version: "2025-05-01-preview" # Updated to the latest version
      index_name: ${SEARCH_RAG_INDEX_NAME} # Name of the search index to use

  - name: primary_postgres_service
    service_catalog_id: azure_postgres_service_01
    settings:
      host: ${POSTGRES_HOST}
      username: ${POSTGRES_USERNAME}
      password: ${POSTGRES_PASSWORD}
      sslmode: ${POSTGRES_SSLMODE}
      auth_type: ${POSTGRES_SERVICE_CREDENTIAL_TYPE} # Valid values are password, azure_key_credential or default_azure_credential
      database: ${POSTGRES_DATABASE_NAME} # Name of the PostgreSQL database to use
      dimensions: 3072 # Dimension of the vector field
      initialize: false # Whether to initialize the database and create the table if it doesn't exist

source_instances:
  - name: sharepoint_01
    source_catalog_id: sharepoint_01 # Reference to source in catalog
    enabled: false
    settings:
      reindex : true
      account_name: ${STORAGE_ACCOUNT_NAME}
      credential_type: ${STORAGE_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      credential_key: ${STORAGE_ACCOUNT_KEY}
  
  - name: primary_blob_storage
    source_catalog_id: azure_storage_01 # Reference to source in catalog
    enabled: true
    settings:
      container_name: "graphrag"
      prefix: "input"
      reindex : true
      account_name: ${STORAGE_ACCOUNT_NAME}
      credential_type: ${STORAGE_ACCOUNT_CREDENTIAL_TYPE} # Valid values are azure_key_credential or default_azure_credential
      credential_key: ${STORAGE_ACCOUNT_KEY}

########################################################
# PIPELINES
# Pipeline definitions - references to step catalog with custom settings
pipelines:

  ###########################################################
  # PIPELINE INSTANCE
  - name: pipeline_1
    description: 'A pipeline to extract, transform, and load documents from Azure Blob Storage to a database'
    version: '1.0'
    #run every 5 mins
    schedule: "*/2 * * * *" # Run every 5 minutes
    crawl_schedule: "*/2 * * * *" # Run every 5 minutes
    purge_schedule: "*/2 * * * *" # Run every 5 minutes

    ###########################################################
    # PIPELINE STEP INSTANCES
    # Define the steps in the pipeline, referencing the step catalog and providing custom settings
    steps:
      ## STEP 1: Sample Step
      - name: sample_step_1
        step_catalog_id: sample_step # Reference to step in catalog
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          key1: "custom_value1"
          key2: "custom_value2"

      ## Flair Step
      - name: flair
        step_catalog_id: flair_step # Reference to step in catalog
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          key1: "custom_value1"
          key2: "custom_value2"

      ## Graphrag Step
      - name: graphrag
        step_catalog_id: graphrag_step # Reference to step in catalog
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          key1: "custom_value1"
          key2: "custom_value2"

      ## STEP 2: Extract Content
      - name: "content_extractor"
        step_catalog_id: content_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          identification_methods: "magic_bytes, file_extension" # Methods to use for document type identification, comma separated list of methods to use, available methods: magic_bytes, file_extension

      ## STEP 2: Extract Content
      - name: "content_chunker"
        step_catalog_id: content_chunker
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:

      ## STEP 2: Extract Content
      - name: "content_embed"
        step_catalog_id: content_embed
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          ai_model_embed_service: primary_ai_embedding_service # Reference to the AI model embed service instance
      
      ## STEP 2: Document Type Identifier
      - name: "document_type_identifier_1"
        step_catalog_id: document_type_identifier
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          identification_methods: "magic_bytes, file_extension" # Methods to use for document type identification, comma separated list of methods to use, available methods: magic_bytes, file_extension
    
      ## STEP 3.1: PDF Text Extractor
      - name: pdf_text_extractor_1
        step_catalog_id: pdf_text_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_blob_storage, primary_ai_inference_service] # References to service instances
        condition: "document_type.primary_type == 'pdf'" # Only run this step if the document type is PDF
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          png_output_folder: "./output/png"
          num_pages: 1 # Maximum number of pages to convert (-1 = all pages)
          prompts:
            system: "You are an AI assistant that helps convert images of pages of a pdf document to markdown text. Only output valid markdown."
            user: |
              Extract the text from the following image into markdown and provide descriptions of images. 
              Always format the markdown as follows to distinguish the text extracted from image descriptions:
              
              ==Extracted-Text==
              {Insert extracted text as markdown here}
              ==End-Extracted-Text==

              ==Image-Descriptions==
              {Insert image descriptions as markdown here}
              ==End-Image-Descriptions==
          max_completion_tokens: 4000
          temperature: 1.0
          top_p: 1.0
          frequency_penalty: 0.0
          presence_penalty: 0.0

      ## STEP 3.2: WORD Text Extractor
      - name: word_text_extractor_1
        step_catalog_id: word_text_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_ai_inference_service] # References to service instances
        condition: "document_type.primary_type == 'word_document'" # Only run this step if the document type is Word document
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: true # Enable debug mode for this step
        settings:
          png_output_folder: "./output/png"
          extract_images: false # Whether to extract text from images in slides
          extract_image_descriptions: false # Whether to extract descriptions from images in slides
          extract_tables: false # Whether to extract tables from slides
          max_chunk_size: 4000 # Maximum number of chars in each chunk
          prompts:
            system: "You are an AI assistant that helps convert images extracted from a word document to markdown text. Only output valid markdown."
            user: |
              Extract the text from the following image into markdown and provide descriptions of images. If the image has no text, don't output any text, just provide the image description. 
              Always format the markdown as follows to distinguish the text extracted from image descriptions:
              
              ==Extracted-Text==
              {Insert extracted text as markdown here}
              ==End-Extracted-Text==

              ==Image-Descriptions==
              {Insert image descriptions as markdown here}
              ==End-Image-Descriptions==
          max_completion_tokens: 4000
          temperature: 1.0
          top_p: 1.0
          frequency_penalty: 0.0
          presence_penalty: 0.0

      ## STEP 3.3: PPTX Text Extractor
      - name: pptx_text_extractor_1
        step_catalog_id: pptx_text_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_ai_inference_service] # References to service instances
        condition: "document_type.primary_type == 'powerpoint_presentation'" # Only run this step if the document type is PowerPoint presentation
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          png_output_folder: "./output/png"
          num_slides: 3 # Maximum number of slides to convert (-1 = all slides)
          extract_images: false # Whether to extract text from images in slides
          extract_image_descriptions: false # Whether to extract descriptions from images in slides
          extract_shapes: false # Whether to extract shapes from slides
          extract_tables: false # Whether to extract tables from slides
          prompts:
            system: "You are an AI assistant that helps convert images extracted from a pptx document to markdown text. Only output valid markdown."
            user: |
              Extract the text from the following image into markdown and provide descriptions of images. If the image has no text, don't output any text, just provide the image description. 
              Always format the markdown as follows to distinguish the text extracted from image descriptions:
              
              ==Extracted-Text==
              {Insert extracted text as markdown here}
              ==End-Extracted-Text==

              ==Image-Descriptions==
              {Insert image descriptions as markdown here}
              ==End-Image-Descriptions==
          max_completion_tokens: 4000
          temperature: 1.0
          top_p: 1.0
          frequency_penalty: 0.0
          presence_penalty: 0.0

      ## STEP 3.4: Excel Text Extractor
      - name: excel_text_extractor_1
        step_catalog_id: excel_text_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        condition: "document_type.primary_type == 'excel_spreadsheet'" # Only run this step if the document type is Excel spreadsheet
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          png_output_folder: "./output/png"
          extract_images: true # Whether to extract text from images in sheets
          extract_charts: true # Whether to extract charts from sheets
          max_rows_per_sheet: -1 # Maximum number of rows to extract from each sheet (-1 = all rows)
          max_columns_per_sheet: -1 # Maximum number of columns to extract from each sheet (-1 = all columns)
          sheets_to_process: [] # Comma separated list of sheets to process, empty list means all sheets


      ## STEP 4: Document Intellegence Extraction
      - name: document_intelligence_extractor_for_pdf
        step_catalog_id: azure_document_intelligence_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_document_intelligence_service] # References to service instances
        condition: "document_type.primary_type in ['pdf']" # Only run this step if the document type is PDF
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: true # Enable debug mode for this step
        settings:
          model_id: "prebuilt-layout" # Model to use for document intelligence extraction
          extract_tables: false # Whether to extract tables from documents
          extract_key_value_pairs: false # Whether to extract key-value pairs from documents
          extract_paragraphs: false # Whether to extract paragraphs from documents
          chunk_by_pages: true # Whether to chunk the content by pages
          output_format: "markdown" # Format for the extracted content

      - name: document_intelligence_extractor_for_pptx
        step_catalog_id: azure_document_intelligence_extractor
        enabled: true
        fail_pipeline_on_error: true
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_document_intelligence_service] # References to service instances
        condition: "document_type.primary_type in ['powerpoint_presentation']" # Only run this step if the document type is PowerPoint
        fail_step_on_document_error: true # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: true # Enable debug mode for this step
        settings:
          model_id: "prebuilt-layout" # Model to use for document intelligence extraction
          extract_tables: false # Whether to extract tables from documents
          extract_key_value_pairs: false # Whether to extract key-value pairs from documents
          extract_paragraphs: false # Whether to extract paragraphs from documents
          chunk_by_pages: false # Whether to chunk the content by pages
          output_format: "markdown" # Format for the extracted content


      ## STEP 5: Custom AI Prompt
      - name: custom_ai_prompt_1
        step_catalog_id: custom_ai_prompt
        enabled: false
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_ai_inference_service] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          chunk_field_to_apply_prompt_on: markdown_text
          output_field_name: custom_ai_prompt_output
          prompts:
            system: "You are an AI assistant. Please process the input accordingly."
            user: "This is a custom AI prompt step. Please process the input accordingly.\nInput: {chunk_content}"
          max_completion_tokens: 4000
          temperature: 1.0
          top_p: 1.0
          frequency_penalty: 0.0
          presence_penalty: 0.0

      ## STEP 5: AI Search Index Writer
      - name: ai_search_index_writer_1
        step_catalog_id: ai_search_index_writer
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_ai_search_service] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          index_name: "${SEARCH_RAG_INDEX_NAME}"
          index_field_mappings: | 
                    {
                    "id": "id",
                    "id": "parent_id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page_num": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "file_path": "filepath",
                    "url": "url",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "content_vector": "content_vector",
                    "caption_vector": "caption_vector"
                    }
      
      ## STEP 5: Postgres Index Writer
      - name: postgres_index_writer_1
        step_catalog_id: postgres_index_writer
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_postgres_service] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          collection_name: "vectorstore"
          database: "${POSTGRES_DATABASE_NAME}"
          properties:
            azure_ml_scoring_endpoint: "${AI_FOUNDRY_ACCOUNT_ENDPOINT}"
            azure_ml_endpoint_key: "${AI_FOUNDRY_ACCOUNT_APIKEY}"
            azure_ml_deployment: "documents"
            azure_openai_endpoint: "id"
            azure_openai_endpoint_key: "id"
            azure_openai_deployment: "id"
          index_field_mappings: | 
                    {
                    "id": "id",
                    "id": "parent_id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page_num": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "file_path": "filepath",
                    "url": "url",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "content_vector": "content_vector",
                    "caption_vector": "caption_vector"
                    }
      
      ## STEP 5: AI Search Index Writer
      - name: ai_search_purge
        step_catalog_id: ai_search_purge
        enabled: true
        fail_pipeline_on_error: false
        retry_on_failure: false
        retries: 3
        timeout: 600
        services: [primary_ai_search_service] # References to service instances
        condition:
        fail_step_on_document_error: false # Fail the step if document processing fails, this is useful for debugging, if set to false, the pipeline will continue even if this step fails
        debug_mode: false # Enable debug mode for this step
        settings:
          index_name: "${SEARCH_RAG_INDEX_NAME}"
          index_field_mappings: | 
                    {
                    "id": "id",
                    "id": "parent_id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page_num": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "file_path": "filepath",
                    "url": "url",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "content_vector": "content_vector",
                    "caption_vector": "caption_vector"
                    }

      
    ##########################################################
    # PIPELINE EXECUTION
    # Define the execution sequence of the steps in the pipeline
    ## execution_sequence: [document_type_identifier_1, pdf_text_extractor_1, word_text_extractor_1, pptx_text_extractor_1, excel_text_extractor_1, custom_ai_prompt_1, ai_search_index_writer_1]
    execution_sequence: [content_extractor, document_type_identifier_1, document_intelligence_extractor_for_pdf, document_intelligence_extractor_for_pptx,content_chunker, flair, graphrag, content_embed,postgres_index_writer_1]

    ##########################################################
    # PIPELINE SETTINGS
    # Global settings for the pipeline execution
    settings:
      enabled: true
      retry_delay: 5
      timeout: 300
      max_concurrent_runs: 5

  
"""

await save_prompt("pipeline_config", pipeline_config, container="docproc")

In [ ]:
service_catalog = decode_prompt('service_catalog', container="docproc")
print(service_catalog)

In [ ]:
service_catalog = """

services_catalog:

  # Sample service configuration
  - id: sample_service_01                                                       # Unique identifier for the service
    name: "Sample Service"                                                      # Human-readable name for the service
    description: "A sample service configuration demonstrating all properties"  # Description of the service
    type: sample_type                                                           # The type/category of the service
    module_name: sample_service                                                 # Python module name implementing the service
    module_path: ./doc/proc/service/sample_service.py                           # Path to the module file
    class_name: SampleService                                                   # Class name to instantiate for this service
    test_connection: false                                                      # Whether to test connection on startup
    category: "Sample Category"                                                 # Logical grouping/category for the service
    version: "1.0"                                                              # Version of the service definition
    tags: [sample, demo, test]                                                  # List of tags for search/filtering

    # Schema for required and optional settings
    settings_schema:
      sample_setting:
        type: string
        title: "Sample Setting"
        description: "A sample setting required by the service"
        required: true
        env_var: "SAMPLE_SERVICE_SETTING"
        default: ${SAMPLE_SERVICE_SETTING}
      sample_optional:
        type: integer
        title: "Optional Setting"
        description: "An optional integer setting"
        required: false
        default: 10
        min: 0  # Minimum value for the integer
        max: 100  # Maximum value for the integer
      sample_with_pattern:
        type: string
        title: "Pattern Setting"
        description: "A setting that must match a specific pattern"
        required: true
        pattern: "^[a-zA-Z0-9_]+$"  # Must be alphanumeric or underscore
        env_var: "SAMPLE_SERVICE_PATTERN_SETTING"
        default: ${SAMPLE_SERVICE_PATTERN_SETTING}
      sample_enum:
        type: string
        title: "Enum Setting"
        description: "A setting that must be one of the predefined values"
        required: false
        enum: ["option1", "option2", "option3"] # Must be one of these options
        default: "option1"
      sample_sensitive:
        type: string      # Sensitive information, e.g., API keys
        required: true
        sensitive: true  # Marked as sensitive to hide in UI
        title: "Sensitive Setting"
        description: "A sensitive setting that should not be displayed in plain text"
        env_var: "SAMPLE_SERVICE_SENSITIVE_SETTING"  # Environment variable for sensitive data
        default: ${SAMPLE_SERVICE_SENSITIVE_SETTING}  # Use environment variable for sensitive data
      sample_boolean:
        type: boolean
        title: "Boolean Setting"
        description: "A boolean setting to enable or disable a feature"
        required: false
        default: false  # Default value for the boolean setting

    # UI metadata for display purposes
    ui_metadata:
      icon: "flask-round"    # Icon to represent the service in UI
      color: "#CCCCCC"       # Color for the service in UI
      description_short: "A sample service" # Short description for quick reference
      description_long: "This is a sample service configuration used for demonstration and documentation purposes."  # Long description for detailed view
  

  # Azure Blob Storage service for document storage and retrieval
  - id: azure_storage_01
    name: "Azure Blob Storage - Primary"
    description: "Primary Azure Blob Storage account for document storage and retrieval"
    type: azure_blob
    module_name: blob_service
    module_path: ./doc/proc/service/blob_service.py
    class_name: BlobService
    test_connection: true
    category: "Storage"
    version: "1.0"
    tags: [azure, storage, blob, documents]
    
    # Service configuration settings schema
    settings_schema:
      account_name:
        type: string
        title: "Storage Account Name"
        description: "Name of the Azure Blob Storage account"
        required: true
        env_var: "STORAGE_ACCOUNT_NAME"
        default: ${STORAGE_ACCOUNT_NAME}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "STORAGE_ACCOUNT_CREDENTIAL_TYPE"
        default: ${STORAGE_ACCOUNT_CREDENTIAL_TYPE}
      credential_key:
        type: string
        title: "Account Key"
        description: "Account key for Azure Blob Storage authentication"
        required: false
        sensitive: true
        env_var: "STORAGE_ACCOUNT_KEY"
        default: ${STORAGE_ACCOUNT_KEY}
        
    # UI metadata
    ui_metadata:
      icon: "container"
      color: "#0078D4"
      description_short: "Azure Blob Storage for document storage"
      description_long: "Azure Blob Storage service for storing and retrieving documents, images, and other unstructured data."


  # Azure AI Inference Service for document processing and inference
  - id: azure_ai_inference_service_01
    name: "Azure AI Inference Service"
    description: "Azure AI service for document processing and inference"
    type: azure_ai_inference
    module_name: azure_ai_inference_service
    module_path: ./doc/proc/service/azure_ai_inference_service.py
    class_name: AzureAIInferenceService
    test_connection: true
    category: "AI Services"
    version: "1.0"
    tags: [azure, ai, inference, llm, processing]

    settings_schema:
      endpoint:
        type: string
        title: "Service Endpoint"
        description: "Endpoint URL for the Azure AI Inference Service"
        required: true
        pattern: "^https://.*"
        env_var: "AI_FOUNDRY_ACCOUNT_ENDPOINT"
        default: ${AI_FOUNDRY_ACCOUNT_ENDPOINT}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE"
        default: ${AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE}
      api_key:
        type: string
        title: "API Key"
        description: "API key for Azure AI Inference Service authentication"
        required: false
        sensitive: true
        env_var: "AI_FOUNDRY_ACCOUNT_APIKEY"
        default: ${AI_FOUNDRY_ACCOUNT_APIKEY}
      model_name:
        type: string
        title: "Model Name"
        description: "Name of the AI model to use for inference"
        default: "chat"
        #enum: ["chat"]
      max_tokens:
        type: integer
        title: "Max Tokens"
        description: "Maximum number of tokens for responses"
        default: 4000
        minimum: 100
        maximum: 8000
    
    ui_metadata:
      icon: "brain"
      color: "#8B5CF6"
      description_short: "Azure AI Inference for document processing"
      description_long: "Azure AI Inference service for natural language processing, document analysis, and content generation."

  # Azure AI Embedding Service for document processing and inference
  - id: azure_ai_embedding_service_01
    name: "Azure AI Embedding Service"
    description: "Azure AI service for document embedding and semantic search"
    type: azure_ai_embedding
    module_name: azure_ai_embedding_service
    module_path: ./doc/proc/service/azure_ai_embedding_service.py
    class_name: AzureAIEmbeddingService
    test_connection: true
    category: "AI Services"
    version: "1.0"
    tags: [azure, ai, embedding, processing]

    settings_schema:
      endpoint:
        type: string
        title: "Service Endpoint"
        description: "Endpoint URL for the Azure AI Embedding Service"
        required: true
        pattern: "^https://.*"
        env_var: "AI_FOUNDRY_ACCOUNT_ENDPOINT"
        default: ${AI_FOUNDRY_ACCOUNT_ENDPOINT}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE"
        default: ${AI_FOUNDRY_ACCOUNT_CREDENTIAL_TYPE}
      api_key:
        type: string
        title: "API Key"
        description: "API key for Azure AI Embedding Service authentication"
        required: false
        sensitive: true
        env_var: "AI_FOUNDRY_ACCOUNT_APIKEY"
        default: ${AI_FOUNDRY_ACCOUNT_APIKEY}
      model_name:
        type: string
        title: "Model Name"
        description: "Name of the AI model to use for embedding"
        default: "text-embedding"
      max_tokens:
        type: integer
        title: "Max Tokens"
        description: "Maximum number of tokens for responses"
        default: 4000
        minimum: 100
        maximum: 8000
    
    ui_metadata:
      icon: "brain"
      color: "#8B5CF6"
      description_short: "Azure AI Embedding for document processing"
      description_long: "Azure AI Embedding service for natural language processing, document analysis, and content generation."

  # Azure Document Intelligence Service for advanced document analysis
  - id: azure_document_intelligence_service_01
    name: "Azure Document Intelligence Service"
    description: "Azure Document Intelligence service for advanced document analysis and content extraction"
    type: azure_document_intelligence
    module_name: azure_document_intelligence_service
    module_path: ./doc/proc/service/azure_document_intelligence_service.py
    class_name: AzureDocumentIntelligenceService
    test_connection: true
    category: "AI Services"
    version: "1.0"
    tags: [azure, ai, document, intelligence, forms, content, understanding]

    settings_schema:
      endpoint:
        type: string
        title: "Service Endpoint"
        description: "Endpoint URL for the Azure Document Intelligence Service"
        required: true
        pattern: "^https://.*"
        env_var: "AZURE_DOCUMENT_INTELLIGENCE_SERVICE_ENDPOINT"
        default: ${AZURE_DOCUMENT_INTELLIGENCE_SERVICE_ENDPOINT}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "AZURE_DOCUMENT_INTELLIGENCE_SERVICE_CREDENTIAL_TYPE"
        default: ${AZURE_DOCUMENT_INTELLIGENCE_SERVICE_CREDENTIAL_TYPE}
      api_key:
        type: string
        title: "API Key"
        description: "API key for Azure Document Intelligence Service authentication"
        required: false
        sensitive: true
        env_var: "AZURE_DOCUMENT_INTELLIGENCE_SERVICE_APIKEY"
        default: ${AZURE_DOCUMENT_INTELLIGENCE_SERVICE_APIKEY}
      model_id:
        type: string
        title: "Model ID"
        description: "ID of the AI model to use. Can be any model supported by Azure Document Intelligence, either prebuilt or custom."
        default: "prebuilt-layout"
        enum: ["prebuilt-layout", "prebuilt-read", 'prebuilt-read', 'prebuilt-businessCard', 'prebuilt-idDocument', 'prebuilt-invoice', 'prebuilt-receipt', 'prebuilt-tax.us.w2', 'prebuilt-healthInsuranceCard.us']
    
    ui_metadata:
      icon: "brain-circuit"
      color: "#8B5CF6"
      description_short: "Azure AI Document Intelligence for document processing"
      description_long: "Azure AI Document Intelligence service for natural language processing, document analysis, and content extraction."


  # Azure AI Search Service for indexing and searching documents
  - id: azure_ai_search_service_01
    name: "Azure AI Search Service"
    description: "Azure AI Search service for indexing and searching documents"
    type: azure_ai_search
    module_name: azure_ai_search_service
    module_path: ./doc/proc/service/azure_ai_search_service.py
    class_name: AzureAISearchService
    test_connection: false
    category: "Search"
    version: "1.0"
    tags: [azure, search, index, cognitive]
    
    settings_schema:
      account_name:
        type: string
        title: "Search Service Account Name"
        description: "Name of the Azure AI Search service"
        required: true
        env_var: "AZURE_SEARCH_SERVICE_ACCOUNT_NAME"
        default: ${AZURE_SEARCH_SERVICE_ACCOUNT_NAME}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "AZURE_AI_SEARCH_SERVICE_CREDENTIAL_TYPE"
        default: ${AZURE_AI_SEARCH_SERVICE_CREDENTIAL_TYPE}
      api_key:
        type: string
        title: "Admin API Key"
        description: "Admin API key for Azure AI Search service"
        required: true
        sensitive: true
        env_var: "AZURE_SEARCH_SERVICE_APIKEY"
        default: ${AZURE_SEARCH_SERVICE_APIKEY}
      api_version:
        type: string
        title: "API Version"
        description: "Azure Search API version to use"
        default: "2025-05-01-preview"
        enum: ['2025-05-01-preview', '2024-07-01', '2023-11-01']
      index_name:
        type: string
        title: "Index Name"
        description: "Name of the search index to use"
        required: true
        env_var: "SEARCH_RAG_INDEX_NAME"
        default: "${SEARCH_RAG_INDEX_NAME}"
    
    ui_metadata:
      icon: "search"
      color: "#FF6B35"
      description_short: "Azure AI Search for document indexing and retrieval"
      description_long: "Azure AI Search service for full-text search, semantic search, and vector search capabilities."

  # Azure PostgreSQL Service for managing database operations
  - id: azure_postgres_service_01
    name: "PostgreSQL Service"
    description: "PostgreSQL service for managing database operations"
    type: azure_postgres
    module_name: azure_postgres_service
    module_path: ./doc/proc/service/postgres_service.py
    class_name: PostgresService
    test_connection: true
    category: "Database"
    version: "1.0"
    tags: [azure, postgres, database]
    
    settings_schema:
      collection_name:
        type: string
        title: "PostgreSQL Collection Name"
        description: "Name of the collection in the Azure PostgreSQL database"
        required: true
        env_var: "POSTGRES_HOST"
        default: "vectorstore"
      host:
        type: string
        title: "PostgreSQL Server Name"
        description: "Name of the Azure PostgreSQL server"
        required: true
        env_var: "POSTGRES_HOST"
        default: ${POSTGRES_HOST}
      username:
        type: string
        title: "PostgreSQL Username"
        description: "Username for the Azure PostgreSQL server"
        required: true
        env_var: "POSTGRES_USERNAME"
        default: ${POSTGRES_USERNAME}
      password:
        type: string
        title: "PostgreSQL Password"
        description: "Password for the Azure PostgreSQL server"
        required: true
        env_var: "POSTGRES_PASSWORD"
        default: ${POSTGRES_PASSWORD}
      sslmode:
        type: string
        title: "PostgreSQL SSL Mode"
        description: "SSL mode for the Azure PostgreSQL server"
        required: true
        env_var: "POSTGRES_SSLMODE"
        default: ${POSTGRES_SSLMODE}
      auth_type:
        type: string
        title: "Authentication Type"
        description: "Type of authentication used for connecting to the database"
        default: "azure_key_credential"
        enum: ["password", "azure_key_credential", "default_azure_credential"]
        env_var: "POSTGRES_SERVICE_CREDENTIAL_TYPE"
        default: ${POSTGRES_SERVICE_CREDENTIAL_TYPE}
      database:
        type: string
        title: "Database Name"
        description: "Name of the PostgreSQL database to use"
        required: true
        env_var: "POSTGRES_DATABASE_NAME"
        default: "${POSTGRES_DATABASE_NAME}"

    ui_metadata:
      icon: "search"
      color: "#FF6B35"
      description_short: "Postgres for document indexing and retrieval"
      description_long: "Postgres service for full-text search, semantic search, and vector search capabilities."



  # Azure Cosmos DB for storing processed document metadata and results
  - id: azure_cosmos_db_01
    name: "Azure Cosmos DB"
    description: "Azure Cosmos DB for storing processed document metadata and results"
    type: azure_cosmos
    module_name: blob_service
    module_path: ./doc/proc/service/blob_service.py
    class_name: BlobService
    test_connection: true
    category: "Database"
    version: "1.0"
    tags: [azure, cosmos, database, nosql, metadata]
    
    settings_schema:
      endpoint:
        type: string
        title: "Cosmos DB Endpoint"
        description: "Endpoint URL for Azure Cosmos DB"
        required: true
        pattern: "^https://.*"
        env_var: "AZURE_COSMOS_DB_ENDPOINT"
        default: ${AZURE_COSMOS_DB_ENDPOINT}
      key:
        type: string
        title: "Primary Key"
        description: "Primary key for Cosmos DB authentication"
        required: true
        sensitive: true
        env_var: "AZURE_COSMOS_DB_KEY"
        default: ${AZURE_COSMOS_DB_KEY}
      database_name:
        type: string
        title: "Database Name"
        description: "Name of the Cosmos DB database"
        required: true
        default: "document-processing"
      container_name:
        type: string
        title: "Container Name"
        description: "Name of the Cosmos DB container"
        required: true
        default: "documents"
      partition_key:
        type: string
        title: "Partition Key"
        description: "Partition key for the container"
        default: "/id"
    
    ui_metadata:
      icon: "database"
      color: "#00BCF2"
      description_short: "Azure Cosmos DB for document metadata"
      description_long: "Azure Cosmos DB service for storing document metadata, processing results, and application state."


  # Azure Content Understanding Service for advanced document analysis
  - id: azure_content_understanding_service_01
    name: "Azure Content Understanding Service"
    description: "Azure Content Understanding service for advanced document analysis and content extraction"
    type: azure_content_understanding
    module_name: azure_content_understanding_service
    module_path: ./doc/proc/service/azure_content_understanding_service.py
    class_name: AzureContentUnderstandingService
    test_connection: true
    category: "AI Services"
    version: "1.0"
    tags: [azure, ai, document, intelligence, forms, content, understanding]

    settings_schema:
      endpoint:
        type: string
        title: "Service Endpoint"
        description: "Endpoint URL for the Azure Content Understanding Service"
        required: true
        pattern: "^https://.*"
        env_var: "AZURE_CONTENT_UNDERSTANDING_SERVICE_ENDPOINT"
        default: ${AZURE_CONTENT_UNDERSTANDING_SERVICE_ENDPOINT}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "AZURE_CONTENT_UNDERSTANDING_SERVICE_CREDENTIAL_TYPE"
        default: ${AZURE_CONTENT_UNDERSTANDING_SERVICE_CREDENTIAL_TYPE}
      api_key:
        type: string
        title: "API Key"
        description: "API key for Azure Content Understanding Service authentication"
        required: false
        sensitive: true
        env_var: "AZURE_CONTENT_UNDERSTANDING_SERVICE_APIKEY"
        default: ${AZURE_CONTENT_UNDERSTANDING_SERVICE_APIKEY}
      api_version:
        type: string
        title: "API Version"
        description: "API version to use for the service"
        default: "2023-07-31"
        enum: ["2023-07-31", "2022-08-31", "2021-09-30-preview"]
      model_id:
        type: string
        title: "Default Model ID"
        description: "Default model ID to use for document analysis"
        default: "prebuilt-layout"
        enum: ["prebuilt-layout", "prebuilt-document", "prebuilt-read", "prebuilt-businessCard", "prebuilt-idDocument", "prebuilt-invoice", "prebuilt-receipt", "prebuilt-tax.us.w2", "prebuilt-healthInsuranceCard.us"]

    ui_metadata:
      icon: "document-text"
      color: "#FF6900"
      description_short: "Azure Content Understanding for document analysis"
      description_long: "Azure Content Understanding service for extracting text, tables, key-value pairs, and structured content from various document formats including PDFs, images, and Office documents."
"""

await save_prompt("service_catalog", service_catalog, container="docproc")

In [ ]:
source_catalog = decode_prompt('source_catalog', container="docproc")
print(source_catalog)

In [ ]:
source_catalog = """
# source_catalog.yaml - Reusable source definitions

sources_catalog:

  # SharePoint source
  - id: sharepoint_01
    name: "SharePoint - Primary"
    description: "Primary SharePoint site for document storage and retrieval"
    type: sharepoint
    module_name: doc.proc.source.sharepoint
    module_path: ./doc/proc/source/sharepoint.py
    class_name: SharePointSource
    test_connection: false
    enabled: false
    category: "Storage"
    version: "1.0"
    tags: [storage, sharepoint, documents]

    # Source configuration settings schema
    settings_schema:

  # Azure Blob Storage source
  - id: azure_storage_01
    name: "Azure Blob Storage - Primary"
    description: "Primary Azure Blob Storage account for document storage and retrieval"
    type: azure_blob
    module_name: doc.proc.source.blob
    module_path: ./doc/proc/source/blob.py
    class_name: BlobSource
    container_name: "graphrag"
    test_connection: true
    enabled: false
    category: "Storage"
    version: "1.0"
    tags: [azure, storage, blob, documents]

    # Source configuration settings schema
    settings_schema:
      container_name:
        type: string
        title: "Container Name"
        description: "Name of the Azure Blob Storage container"
        required: true
        env_var: "STORAGE_CONTAINER_NAME"
        default: "documents"
      file_types:
        type: string
        title: "File Types"
        description: "Comma-separated list of file types to include (e.g. .pdf, .docx)"
        required: true
        env_var: "STORAGE_FILE_TYPES"
        default: "pdf,docx"
      paths:
        type: string
        title: "File Paths"
        description: "Comma-separated list of file paths to include (e.g. /path/to/file1.pdf)"
        required: true
        env_var: "STORAGE_FILE_PATHS"
        default: "/path/to/file1.pdf,/path/to/file2.pdf"
      include_metadata:
        type: bool
        title: "Include Metadata"
        description: "Whether to include metadata in the processing"
        required: true
        env_var: "STORAGE_INCLUDE_METADATA"
        default: true
      recursive:
        type: bool
        title: "Recursive"
        description: "Whether to include files from subdirectories"
        required: true
        env_var: "STORAGE_RECURSIVE"
        default: true
      account_name:
        type: string
        title: "Storage Account Name"
        description: "Name of the Azure Blob Storage account"
        required: true
        env_var: "STORAGE_ACCOUNT_NAME"
        default: ${STORAGE_ACCOUNT_NAME}
      credential_type:
        type: string
        title: "Credential Type"
        description: "Type of credential used for authentication"
        default: "azure_key_credential"
        enum: ["azure_key_credential", "default_azure_credential"]
        env_var: "STORAGE_ACCOUNT_CREDENTIAL_TYPE"
        default: ${STORAGE_ACCOUNT_CREDENTIAL_TYPE}
      credential_key:
        type: string
        title: "Account Key"
        description: "Account key for Azure Blob Storage authentication"
        required: false
        sensitive: true
        env_var: "STORAGE_ACCOUNT_KEY"
        default: ${STORAGE_ACCOUNT_KEY}
        
    # UI metadata
    ui_metadata:
      icon: "container"
      color: "#0078D4"
      description_short: "Azure Blob Storage for document storage"
      description_long: "Azure Blob Storage service for storing and retrieving documents, images, and other unstructured data."

  # Azure Cosmos DB for storing processed document metadata and results
  - id: azure_cosmos_db_01
    name: "Azure Cosmos DB"
    description: "Azure Cosmos DB for storing processed document metadata and results"
    type: azure_cosmos
    module_name: doc.proc.source.cosmos
    module_path: ./doc/proc/source/cosmos.py
    class_name: CosmosService
    test_connection: true
    category: "Database"
    version: "1.0"
    tags: [azure, cosmos, database, nosql, metadata]
    
    settings_schema:
      endpoint:
        type: string
        title: "Cosmos DB Endpoint"
        description: "Endpoint URL for Azure Cosmos DB"
        required: true
        pattern: "^https://.*"
        env_var: "AZURE_COSMOS_DB_ENDPOINT"
        default: ${AZURE_COSMOS_DB_ENDPOINT}
      key:
        type: string
        title: "Primary Key"
        description: "Primary key for Cosmos DB authentication"
        required: true
        sensitive: true
        env_var: "AZURE_COSMOS_DB_KEY"
        default: ${AZURE_COSMOS_DB_KEY}
      database_name:
        type: string
        title: "Database Name"
        description: "Name of the Cosmos DB database"
        required: true
        default: "document-processing"
      container_name:
        type: string
        title: "Container Name"
        description: "Name of the Cosmos DB container"
        required: true
        default: "documents"
      partition_key:
        type: string
        title: "Partition Key"
        description: "Partition key for the container"
        default: "/id"
    
    ui_metadata:
      icon: "database"
      color: "#00BCF2"
      description_short: "Azure Cosmos DB for document metadata"
      description_long: "Azure Cosmos DB service for storing document metadata, processing results, and application state."
"""

await save_prompt("source_catalog", source_catalog, container="docproc")

In [ ]:
step_catalog = decode_prompt('step_catalog', container="docproc")
print(step_catalog)

In [ ]:
step_catalog = """
# step_catalog.yaml - Reusable step definitions

step_catalog:

  ########################################################
  # Sample step for development and testing purposes
  - id: sample_step
    name: "Sample Development Step"
    description: "Sample step for development and testing"
    type: script
    module_name: sample
    module_path: ./doc/proc/step/sample.py
    class_name: SampleStep
    tags: [sample, development]
    category: "Development"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:
      key1:
        type: string
        title: "Custom Key 1"
        description: "First custom configuration key"
        default: "value1"
      key2:
        type: string
        title: "Custom Key 2"
        description: "Second custom configuration key"
        default: "value2"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "gear"
      color: "#6B7280"
      description_short: "A sample step used for development and testing purposes."
      description_long: "A sample step used for development and testing purposes. Can be customized with various key-value pairs."

  ########################################################
  # Flair step for Named Entity Recognition (NER)
  - id: flair_step
    name: "Flair NER Step"
    description: "Named Entity Recognition using Flair"
    type: script
    module_name: flair_step
    module_path: ./doc/proc/step/flair.py
    class_name: FlairStep
    tags: [flair, ner, extraction]
    category: "Development"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:
      key1:
        type: string
        title: "Custom Key 1"
        description: "First custom configuration key"
        default: "value1"
      key2:
        type: string
        title: "Custom Key 2"
        description: "Second custom configuration key"
        default: "value2"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "gear"
      color: "#6B7280"
      description_short: "A sample step used for development and testing purposes."
      description_long: "A sample step used for development and testing purposes. Can be customized with various key-value pairs."
  
  ########################################################
  # Graphrag step for document processing
  - id: graphrag_step
    name: "Graphrag Document Processing Step"
    description: "Document processing using Graphrag"
    type: script
    enabled: true
    module_name: graphrag_step
    module_path: ./doc/proc/step/graphrag.py
    class_name: GraphRagStep
    tags: [graphrag, document, processing]
    category: "Development"
    version: "1.0"
    settings:
      key1: "value1"
      key2: "value2"

    # Settings schema for UI generation and validation
    settings_schema:
      key1:
        type: string
        title: "Custom Key 1"
        description: "First custom configuration key"
        default: "value1"
      key2:
        type: string
        title: "Custom Key 2"
        description: "Second custom configuration key"
        default: "value2"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "gear"
      color: "#6B7280"
      description_short: "A sample step used for development and testing purposes."
      description_long: "A sample step used for development and testing purposes. Can be customized with various key-value pairs."
  
  ########################################################
  # Extract Content Step
  - id: content_extractor
    name: "Extract Content"
    description: "Extracts content from various document types"
    type: script
    module_name: content_extractor
    module_path: ./doc/proc/step/content_extractor.py
    class_name: ContentExtractorStep
    tags: [document, extraction, content]
    category: "Processing"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:
      identification_methods:
        type: string
        title: "Identification Methods"
        description: "Methods to use for document type identification. Comma separated list. Possible values: magic_bytes, file_extension"
        default: "magic_bytes, file_extension"
        ui_component: "input"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "file-question-mark"
      color: "#F59E0B"
      description_short: "Identifies document types based on file properties."
      description_long: "This step identifies the type of documents based on various methods such as magic bytes and file extensions. It helps in classifying documents for further processing."  

########################################################
  # Extract Content Step
  - id: content_chunker
    name: "Chunk Content"
    description: "Chunks content from various document types"
    type: script
    module_name: content_chunker
    module_path: ./doc/proc/step/content_chunker.py
    class_name: ContentChunkerStep
    tags: [document, chunking, content]
    category: "Processing"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:
      

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "file-question-mark"
      color: "#F59E0B"
      description_short: "Identifies document types based on file properties."
      description_long: "This step identifies the type of documents based on various methods such as magic bytes and file extensions. It helps in classifying documents for further processing."  

########################################################
  # Embed Content Step
  - id: content_embed
    name: "Embed Content"
    description: "Embeds content from various document types"
    type: script
    module_name: content_embed
    module_path: ./doc/proc/step/content_embed.py
    class_name: ContentEmbedStep
    tags: [document, embedding, content]
    category: "Processing"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:  
      ai_model_embed_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "file-question-mark"
      color: "#F59E0B"
      description_short: "Identifies document types based on file properties."
      description_long: "This step identifies the type of documents based on various methods such as magic bytes and file extensions. It helps in classifying documents for further processing."  


  ########################################################
  # Document type identification Step
  - id: document_type_identifier
    name: "Document Type Identifier"
    description: "Identifies the type of documents based on various methods"
    type: script
    module_name: document_type_identifier
    module_path: ./doc/proc/step/document_type_identifier.py
    class_name: DocumentTypeIdentifierStep
    tags: [document, identification, classification]
    category: "Processing"
    version: "1.0"

    # Settings schema for UI generation and validation
    settings_schema:
      identification_methods:
        type: string
        title: "Identification Methods"
        description: "Methods to use for document type identification. Comma separated list. Possible values: magic_bytes, file_extension"
        default: "magic_bytes, file_extension"
        ui_component: "input"

    # UI metadata for dynamic form generation
    ui_metadata:
      icon: "file-question-mark"
      color: "#F59E0B"
      description_short: "Identifies document types based on file properties."
      description_long: "This step identifies the type of documents based on various methods such as magic bytes and file extensions. It helps in classifying documents for further processing."  


  ########################################################
  # PDF text extraction step          
  - id: pdf_text_extractor
    name: "PDF Text Extractor"
    description: "Extract text and metadata from PDF documents"
    type: script
    module_name: pdf_text_extractor
    module_path: ./doc/proc/step/pdf_text_extractor.py
    class_name: PDFTextExtractorStep
    tags: [pdf, text, extraction, azure_blob]
    category: "Document Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      storage_service:
        type: string
        title: "Storage Service"
        description: "Reference to the storage service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_blob"
      ai_model_inference_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"
      png_output_folder:
        type: string
        title: "PNG Output Folder"
        description: "Directory path where PNG files will be saved"
        default: "./output/png"
        required: true
        #pattern: "^\\.\\/.*"
      num_pages:
        type: integer
        title: "Number of Pages"
        description: "Maximum number of pages to convert (-1 = all pages)"
        default: -1
        min: -1
        max: 200
      system_prompt:
        type: string
        title: "System Prompt"
        description: "Instructions for the AI system"
        default: "You are an AI assistant that helps convert images of pages of a pdf document to markdown text. Only output valid markdown."
        ui_component: "textarea"
      user_prompt:
        type: string
        title: "User Prompt Template"
        description: "Template for user prompts sent to AI"
        default: "Extract the text from the following image into markdown and provide descriptions of images. Always format the markdown as follows to distinguish the text extracted from image descriptions:\n==Extracted-Text==\n{Insert extracted text as markdown here}==End-Extracted-Text==\n\n==Image-Descriptions==\n{Insert image descriptions as markdown here}==End-Image-Descriptions=="
        ui_component: "textarea"
      max_completion_tokens:
        type: integer
        title: "Max Completion Tokens"
        description: "Maximum number of tokens to generate"
        default: 4000
        min: 100
        max: 8000
      temperature:
        type: number
        title: "Temperature"
        description: "Controls randomness in AI responses (0.0 = deterministic, 2.0 = very random)"
        default: 1.0
        min: 0.0
        max: 2.0
        multipleOf: 0.1
      top_p:
        type: number
        title: "Top P"
        description: "Controls diversity of AI responses"
        default: 0.4
        min: 0.0
        max: 1.0
        multipleOf: 0.1
      frequency_penalty:
        type: number
        title: "Frequency Penalty"
        description: "Reduces repetition in AI responses"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      presence_penalty:
        type: number
        title: "Presence Penalty"
        description: "Encourages AI to talk about new topics"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1

    ui_metadata:
      icon: "file-text"
      color: "#10B981"
      description_short: "Extracts text from PDF document pages into markdown."
      description_long: "Extracts text from PDF document pages into PNG image files and then uses Azure AI Inference Service to convert them to markdown."


  ########################################################
  # WORD text extraction step
  - id: word_text_extractor
    name: "WORD Text Extractor"
    description: "Extract text and metadata from WORD documents"
    type: script
    module_name: word_text_extractor
    module_path: ./doc/proc/step/word_text_extractor.py
    class_name: WordTextExtractorStep
    tags: [word, text, extraction]
    category: "Document Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      ai_model_inference_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"
      png_output_folder:
        type: string
        title: "PNG Output Folder"
        description: "Directory path where PNG files will be saved"
        default: "./output/png"
        required: true
        #pattern: "^\\.\\/.*"
      extract_images:
        type: boolean
        title: "Extract Images"
        description: "Whether to extract text from images in slides"
        default: true
      extract_image_desacriptions:
        type: boolean
        title: "Extract Image Descriptions"
        description: "Whether to extract descriptions from images in slides"
        default: true
      extract_tables:
        type: boolean
        title: "Extract Tables"
        description: "Whether to extract tables from slides"
        default: true
      max_chunk_size:
        type: integer
        title: "Max Chunk Size"
        description: "Maximum number of chars per chunk."
        default: 4000
        min: -1
        max: 1000000
      system_prompt:
        type: string
        title: "System Prompt"
        description: "Instructions for the AI system"
        default: "You are an AI assistant that helps convert images of pages from a word document to markdown text. Only output valid markdown."
        ui_component: "textarea"
      user_prompt:
        type: string
        title: "User Prompt Template"
        description: "Template for user prompts sent to AI"
        default: "Extract the text from the following image into markdown and provide descriptions of images. Always format the markdown as follows to distinguish the text extracted from image descriptions:\n==Extracted-Text==\n{Insert extracted text as markdown here}==End-Extracted-Text==\n\n==Image-Descriptions==\n{Insert image descriptions as markdown here}==End-Image-Descriptions=="
        ui_component: "textarea"
      max_completion_tokens:
        type: integer
        title: "Max Completion Tokens"
        description: "Maximum number of tokens to generate"
        default: 4000
        min: 100
        max: 8000
      temperature:
        type: number
        title: "Temperature"
        description: "Controls randomness in AI responses (0.0 = deterministic, 2.0 = very random)"
        default: 1.0
        min: 0.0
        max: 2.0
        multipleOf: 0.1
      top_p:
        type: number
        title: "Top P"
        description: "Controls diversity of AI responses"
        default: 0.4
        min: 0.0
        max: 1.0
        multipleOf: 0.1
      frequency_penalty:
        type: number
        title: "Frequency Penalty"
        description: "Reduces repetition in AI responses"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      presence_penalty:
        type: number
        title: "Presence Penalty"
        description: "Encourages AI to talk about new topics"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1

    ui_metadata:
      icon: "book-text"
      color: "#10B981"
      description_short: "Extracts text from WORD documents into markdown."
      description_long: "Extracts text, tables, shapes from WORD document slides and uses Azure AI Inference Service to convert them to markdown."


  ########################################################
  # PPTX text extraction step
  - id: pptx_text_extractor
    name: "PPTX Text Extractor"
    description: "Extract text and metadata from PPTX presentations"
    type: script
    module_name: pptx_text_extractor
    module_path: ./doc/proc/step/pptx_text_extractor.py
    class_name: PowerPointTextExtractorStep
    tags: [pptx, text, extraction]
    category: "Document Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      ai_model_inference_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"
      png_output_folder:
        type: string
        title: "PNG Output Folder"
        description: "Directory path where PNG files will be saved"
        default: "./output/png"
        required: true
        #pattern: "^\\.\\/.*"
      extract_images:
        type: boolean
        title: "Extract Images"
        description: "Whether to extract text from images in slides"
        default: true
      extract_image_desacriptions:
        type: boolean
        title: "Extract Image Descriptions"
        description: "Whether to extract descriptions from images in slides"
        default: true
      extract_tables:
        type: boolean
        title: "Extract Tables"
        description: "Whether to extract tables from slides"
        default: true
      extract_shapes:
        type: boolean
        title: "Extract Shapes"
        description: "Whether to extract shapes from slides"
        default: true
      num_slides:
        type: integer
        title: "Number of Slides"
        description: "Maximum number of slides to convert (-1 = all slides)"
        default: -1
        min: -1
        max: 200
      system_prompt:
        type: string
        title: "System Prompt"
        description: "Instructions for the AI system"
        default: "You are an AI assistant that helps convert images from a pptx slide to markdown text. Only output valid markdown."
        ui_component: "textarea"
      user_prompt:
        type: string
        title: "User Prompt Template"
        description: "Template for user prompts sent to AI"
        default: "Extract the text from the following image into markdown and provide descriptions of images. Always format the markdown as follows to distinguish the text extracted from image descriptions:\n==Extracted-Text==\n{Insert extracted text as markdown here}==End-Extracted-Text==\n\n==Image-Descriptions==\n{Insert image descriptions as markdown here}==End-Image-Descriptions=="
        ui_component: "textarea"
      max_completion_tokens:
        type: integer
        title: "Max Completion Tokens"
        description: "Maximum number of tokens to generate"
        default: 4000
        min: 100
        max: 8000
      temperature:
        type: number
        title: "Temperature"
        description: "Controls randomness in AI responses (0.0 = deterministic, 2.0 = very random)"
        default: 1.0
        min: 0.0
        max: 2.0
        multipleOf: 0.1
      top_p:
        type: number
        title: "Top P"
        description: "Controls diversity of AI responses"
        default: 0.4
        min: 0.0
        max: 1.0
        multipleOf: 0.1
      frequency_penalty:
        type: number
        title: "Frequency Penalty"
        description: "Reduces repetition in AI responses"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      presence_penalty:
        type: number
        title: "Presence Penalty"
        description: "Encourages AI to talk about new topics"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1

    ui_metadata:
      icon: "presentation"
      color: "#10B981"
      description_short: "Extracts text from PPTX document slides into markdown."
      description_long: "Extracts text, tables, shapes from PPTX document slides and uses Azure AI Inference Service to convert them to markdown."


  ########################################################
  # Excel text extraction step
  - id: excel_text_extractor
    name: "Excel Text Extractor"
    description: "Extract text and metadata from Excel spreadsheets"
    type: script
    module_name: excel_text_extractor
    module_path: ./doc/proc/step/excel_text_extractor.py
    class_name: ExcelTextExtractorStep
    tags: [excel, text, extraction, spreadsheet]
    category: "Document Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      png_output_folder:
        type: string
        title: "PNG Output Folder"
        description: "Directory path where PNG files will be saved"
        default: "./output/png"
        required: true
        #pattern: "^\\.\\/.*"
      extract_images:
        type: boolean
        title: "Extract Images"
        description: "Whether to extract images from sheets"
        default: true
        ui_component: "checkbox"
      extract_charts:
        type: boolean
        title: "Extract Charts"
        description: "Whether to extract charts from sheets"
        default: true
        ui_component: "checkbox"
      max_rows_per_sheet:
        type: integer
        title: "Max Rows per Sheet"
        description: "Maximum number of rows to process per sheet (-1 = all rows)"
        default: -1
        min: -1
        max: 10000
      max_columns_per_sheet:
        type: integer
        title: "Max Columns per Sheet"
        description: "Maximum number of columns to process per sheet (-1 = all columns)"
        default: -1
        min: -1
        max: 500
      sheets_to_process:
        type: string
        title: "Sheets to Process"
        description: "Comma separated list of sheet names to process (empty = all sheets)"
        default: ""
      
    ui_metadata:
      icon: "table"
      color: "#059669"
      description_short: "Extracts text from Excel spreadsheet sheets into markdown."
      description_long: "Extracts text, images, and charts from Excel spreadsheet sheets and uses Azure AI Inference Service to convert them to markdown."

  ########################################################
  # Postgres Index Writer step
  - id: postgres_index_writer
    name: "Postgres Index Writer"
    description: "Writes AI-generated summaries and metadata to Postgres database"
    type: script
    module_name: postgres_index_writer
    module_path: ./doc/proc/step/postgres_index_writer.py
    class_name: PostgresIndexWriterStep
    tags: [ai, search, index, postgres]
    category: "AI Processing"
    version: "1.0"
    
    # Settings schema for UI generation and validation
    settings_schema:
      postgres_service:
        type: string
        title: "Postgres Service"
        description: "Reference to the Postgres service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "postgres"
      database:
        type: string
        title: "Postgres Database Name"
        description: "Name of the Postgres database to write to"
        default: "vectorstore"
        ui_component: "input"
      index_field_mappings:
        type: string
        title: "Index Field Mappings"
        description: "Mappings of document fields to index fields. Document fields are mapped from the StepInputOutput object model."
        default: |
                  {
                    "id": "id",
                    "parent_id": "parent_id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "filepath": "file_path",
                    "url": "file_path",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "content_vector": "content_vector",
                    "caption_vector": "caption_vector"
                  }
        ui_component: "textarea"
      
    ui_metadata:
      icon: "text-search"
      color: "#8B5CF6"
      description_short: "Writes AI-generated summaries and metadata to search index."
      description_long: "Writes AI-generated summaries and metadata to search index. Supports custom field mappings and index names."

  ########################################################
  # AI Search Index Writer step
  - id: ai_search_index_writer
    name: "AI Search Index Writer"
    description: "Writes AI-generated summaries and metadata to search index"
    type: script
    module_name: ai_search_index_writer
    module_path: ./doc/proc/step/ai_search_index_writer.py
    class_name: AISearchIndexWriterStep
    tags: [ai, search, index, azure_search]
    category: "AI Processing"
    version: "1.0"
    
    # Settings schema for UI generation and validation
    settings_schema:
      ai_search_service:
        type: string
        title: "Azure AI Search Service"
        description: "Reference to the Azure AI Search service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_search"
      storage_service:
        type: string
        title: "Storage Service"
        description: "Reference to the storage service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_blob"
      index_name:
        type: string
        title: "Azure AI Index Name"
        description: "Name of the Azure AI Search index to write to"
        default: "pdf-index"
        ui_component: "input"
      index_field_mappings:
        type: string
        title: "Index Field Mappings"
        description: "Mappings of document fields to index fields. Document fields are mapped from the StepInputOutput object model."
        default: |
                  {
                    "id": "id",
                    "parent_id": "id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "filepath": "file_path",
                    "url": "file_path",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "contentVector": "contentVector",
                    "captionVector": "captionVector"
                  }
        ui_component: "textarea"
      
    ui_metadata:
      icon: "text-search"
      color: "#8B5CF6"
      description_short: "Writes AI-generated summaries and metadata to search index."
      description_long: "Writes AI-generated summaries and metadata to search index. Supports custom field mappings and index names."
  
  ########################################################
  # AI Search Index Writer step
  - id: ai_search_purge
    name: "AI Search Index Purger"
    description: "Purges AI-generated summaries and metadata from search index"
    type: script
    module_name: ai_search_purge
    module_path: ./doc/proc/step/ai_search_purge.py
    class_name: AISearchPurgeStep
    tags: [ai, search, index, azure_search]
    category: "AI Processing"
    version: "1.0"
    
    # Settings schema for UI generation and validation
    settings_schema:
      ai_search_service:
        type: string
        title: "Azure AI Search Service"
        description: "Reference to the Azure AI Search service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_search"
      storage_service:
        type: string
        title: "Storage Service"
        description: "Reference to the storage service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_blob"
      index_name:
        type: string
        title: "Azure AI Index Name"
        description: "Name of the Azure AI Search index to write to"
        default: "pdf-index"
        ui_component: "input"
      index_field_mappings:
        type: string
        title: "Index Field Mappings"
        description: "Mappings of document fields to index fields. Document fields are mapped from the StepInputOutput object model."
        default: |
                  {
                    "id": "id",
                    "parent_id": "id",
                    "metadata_storage_path": "metadata_storage_path",
                    "metadata_storage_name": "metadata_storage_name",
                    "metadata_storage_last_modified": "metadata_storage_last_modified",
                    "metadata_security_id": "metadata_security_id",
                    "chunk_id": "chunk_id",
                    "content": "content",
                    "imageCaptions": "imageCaptions",
                    "page": "page",
                    "offset": "offset",
                    "length": "length",
                    "title": "title",
                    "category": "category",
                    "filepath": "file_path",
                    "url": "file_path",
                    "summary": "summary",
                    "relatedImages": "relatedImages",
                    "relatedFiles": "relatedFiles",
                    "source": "source",
                    "contentVector": "contentVector",
                    "captionVector": "captionVector"
                  }
        ui_component: "textarea"
      
    ui_metadata:
      icon: "text-search"
      color: "#8B5CF6"
      description_short: "Writes AI-generated summaries and metadata to search index."
      description_long: "Writes AI-generated summaries and metadata to search index. Supports custom field mappings and index names."


  ########################################################
  # Custom AI Prompt step
  - id: custom_ai_prompt
    name: "Custom AI Prompt Step"
    description: "Custom step to execute AI prompts with dynamic configuration"
    type: script
    module_name: custom_ai_prompt
    module_path: ./doc/proc/step/custom_ai_prompt.py
    class_name: CustomAIPromptStep
    tags: [ai, prompt, custom]
    category: "AI Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      ai_model_inference_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"
      chunk_field_to_apply_prompt_on:
        type: string
        title: "Field to Apply Prompt On"
        description: "The field key in document.chunks.chunk which the AI prompt will be applied to"
        default: "markdown_text"
        ui_component: "input"
      output_field_name:
        type: string
        title: "Output Field Name"
        description: "Field name in document.chunks.chunk to store the AI response"
        default: "custom_ai_prompt_output"
        ui_component: "input"
      system_prompt:
        type: string
        title: "System Prompt"
        description: "Instructions for the AI system"
        default: "You are an AI assistant."
        ui_component: "textarea"
      user_prompt:
        type: string
        title: "User Prompt Template"
        description: "Template for the custom user prompts sent to AI"
        default: "This is a custom AI prompt step. Please process the input accordingly.\nInput: {chunk_content}"
        ui_component: "textarea"
      max_completion_tokens:
        type: integer
        title: "Max Completion Tokens"
        description: "Maximum number of tokens to generate"
        default: 4000
        min: 100
        max: 8000
      temperature:
        type: number
        title: "Temperature"
        description: "Controls randomness in AI responses (0.0 = deterministic, 2.0 = very random)"
        default: 1.0
        min: 0.0
        max: 2.0
        multipleOf: 0.1
      top_p:
        type: number
        title: "Top P"
        description: "Controls diversity of AI responses"
        default: 0.4
        min: 0.0
        max: 1.0
        multipleOf: 0.1
      frequency_penalty:
        type: number
        title: "Frequency Penalty"
        description: "Reduces repetition in AI responses"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      presence_penalty:
        type: number
        title: "Presence Penalty"
        description: "Encourages AI to talk about new topics"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      
    ui_metadata:
      icon: "brain"
      color: "#8B5CF6"
      description_short: "Writes AI-generated summaries and metadata to search index."
      description_long: "Writes AI-generated summaries and metadata to search index. Supports custom field mappings and index names."


  ########################################################
  # Entity Extractor step
  - id: entity_extractor
    name: "Entity Extractor Step"
    description: "Extract named entities and relationships from document content"
    type: script
    module_name: entity_extractor
    module_path: ./doc/proc/step/entity_extractor.py
    class_name: EntityExtractorStep
    tags: [ai, nlp, entities, relationships, ner]
    category: "AI Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      ai_model_inference_service:
        type: string
        title: "AI Inference Service"
        description: "Reference to the AI service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_ai_inference"
      chunk_field_to_extract_entities_from:
        type: string
        title: "Field to Extract Entities From"
        description: "The field key in document.chunks.chunk from which entities will be extracted"
        default: "text"
        ui_component: "input"
      output_field_name:
        type: string
        title: "Output Field Name"
        description: "Field name in document.chunks.chunk to store the extracted entities"
        default: "extracted_entities"
        ui_component: "input"
      extract_people:
        type: boolean
        title: "Extract People"
        description: "Extract person entities (names of individuals)"
        default: true
        ui_component: "checkbox"
      extract_places:
        type: boolean
        title: "Extract Places"
        description: "Extract place entities (geographical locations, cities, countries)"
        default: true
        ui_component: "checkbox"
      extract_locations:
        type: boolean
        title: "Extract Locations"
        description: "Extract location entities (addresses, buildings, facilities)"
        default: true
        ui_component: "checkbox"
      extract_organizations:
        type: boolean
        title: "Extract Organizations"
        description: "Extract organization entities (companies, institutions)"
        default: true
        ui_component: "checkbox"
      extract_relationships:
        type: boolean
        title: "Extract Relationships"
        description: "Extract relationships between entities"
        default: true
        ui_component: "checkbox"
      custom_entity_types:
        type: string
        title: "Custom Entity Types"
        description: "Comma separated list of custom entity types to extract (e.g., PRODUCT, TECHNOLOGY)"
        default: ""
        ui_component: "input"
      output_format:
        type: string
        title: "Output Format"
        description: "Format for the output data"
        default: "structured"
        enum: ["structured", "json"]
        ui_component: "select"
      include_confidence:
        type: boolean
        title: "Include Confidence Scores"
        description: "Include confidence scores in the output"
        default: true
        ui_component: "checkbox"
      include_context:
        type: boolean
        title: "Include Context"
        description: "Include surrounding text context for entities"
        default: true
        ui_component: "checkbox"
      system_prompt:
        type: string
        title: "System Prompt"
        description: "Instructions for the AI system for entity extraction"
        default: ""
        ui_component: "textarea"
      user_prompt:
        type: string
        title: "User Prompt Template"
        description: "Template for user prompts sent to AI"
        default: ""
        ui_component: "textarea"
      max_completion_tokens:
        type: integer
        title: "Max Completion Tokens"
        description: "Maximum number of tokens to generate"
        default: 4000
        min: 100
        max: 8000
      temperature:
        type: number
        title: "Temperature"
        description: "Controls randomness in AI responses (0.0 = deterministic, 2.0 = very random)"
        default: 0.1
        min: 0.0
        max: 2.0
        multipleOf: 0.01
      top_p:
        type: number
        title: "Top P"
        description: "Controls diversity of AI responses"
        default: 1.0
        min: 0.0
        max: 1.0
        multipleOf: 0.1
      frequency_penalty:
        type: number
        title: "Frequency Penalty"
        description: "Reduces repetition in AI responses"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1
      presence_penalty:
        type: number
        title: "Presence Penalty"
        description: "Encourages AI to talk about new topics"
        default: 0.0
        min: -2.0
        max: 2.0
        multipleOf: 0.1

    ui_metadata:
      icon: "list-tree"
      color: "#10B981"
      description_short: "Extracts named entities and relationships from text."
      description_long: "Uses AI to identify and extract people, places, organizations, and custom entities along with their relationships from document content."


  ########################################################
  # Azure Document Intelligence Extractor Step
  - id: azure_document_intelligence_extractor
    name: "Azure Document Intelligence Extractor"
    description: "Extract structured content from documents using Azure Document Intelligence service"
    type: script
    module_name: azure_document_intelligence_extractor
    module_path: ./doc/proc/step/azure_document_intelligence_extractor.py
    class_name: AzureDocumentIntelligenceExtractorStep
    tags: ["azure", "document intelligence", "extraction"]
    category: "Document Processing"
    version: "1.0"
        
    # Settings schema for UI generation and validation
    settings_schema:
      azure_document_intelligence_service:
        type: string
        title: "Document Intelligence Service"
        description: "Reference to the Azure Document Intelligence service instance to use"
        required: true
        ui_component: "service_selector"
        service_type: "azure_document_intelligence"
      model_id:
        type: string
        title: "Model ID"
        description: "Model ID to use for document analysis"
        default: "prebuilt-layout"
        enum: ["prebuilt-layout", "prebuilt-document", "prebuilt-read", "prebuilt-businessCard", "prebuilt-idDocument", "prebuilt-invoice", "prebuilt-receipt", "prebuilt-tax.us.w2", "prebuilt-healthInsuranceCard.us"]
      extract_tables:
        type: boolean
        title: "Extract Tables"
        description: "Whether to extract and process table content"
        default: true
        ui_component: "checkbox"
      extract_key_value_pairs:
        type: boolean
        title: "Extract Key-Value Pairs"
        description: "Whether to extract key-value pairs from documents"
        default: true
        ui_component: "checkbox"
      extract_paragraphs:
        type: boolean
        title: "Extract Paragraphs"
        description: "Whether to extract paragraph content"
        default: true
        ui_component: "checkbox"
      chunk_by_pages:
        type: boolean
        title: "Chunk by Pages"
        description: "Whether to create separate chunks for each page"
        default: true
        ui_component: "checkbox"
      output_format:
        type: string
        title: "Output Format"
        description: "Format for the extracted content"
        default: "markdown"
        enum: ["structured", "markdown", "json"]
        ui_component: "select"

    ui_metadata:
      icon: "document-search"
      color: "#FF6900"
      description_short: "Extract structured content using Azure Document Intelligence."
      description_long: "Uses Azure Document Intelligence service to extract text, tables, key-value pairs, and structured content from various document formats including PDFs, images, and Office documents."

"""

await save_prompt("step_catalog", step_catalog, container="docproc")

In [ ]:
gr_config = decode_prompt('gr_config', container="docproc")
print(gr_config)

In [ ]:
gr_config = """
### This config file contains required core defaults that must be set, along with a handful of common optional settings.
### For a full list of available settings, see https://microsoft.github.io/graphrag/config/yaml/

### LLM settings ###
## There are a number of settings to tune the threading and token limits for LLM calls - check the docs.

models:
  default_chat_model:
    type: azure_openai_chat # or azure_openai_chat
    api_base: https://aif-REPLACE_ME.openai.azure.com/
    api_version: 2024-05-01-preview
    auth_type: api_key # or azure_managed_identity
    api_key: ${AI_FOUNDRY_ACCOUNT_APIKEY} # set this in the generated .env file
    # audience: "https://cognitiveservices.azure.com/.default"
    # organization: <organization_id>
    model: gpt-4o
    deployment_name: chat
    # encoding_model: cl100k_base # automatically set by tiktoken if left undefined
    model_supports_json: true # recommended if this is available for your model.
    concurrent_requests: 25 # max number of simultaneous LLM requests allowed
    async_mode: threaded # or asyncio
    retry_strategy: native
    max_retries: 10
    tokens_per_minute: auto              # set to null to disable rate limiting
    requests_per_minute: auto            # set to null to disable rate limiting
  default_embedding_model:
    type: azure_openai_embedding # or azure_openai_embedding
    api_base: https://aif-REPLACE_ME.openai.azure.com/
    api_version: 2024-05-01-preview
    auth_type: api_key # or azure_managed_identity
    api_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}
    # audience: "https://cognitiveservices.azure.com/.default"
    # organization: <organization_id>
    model: text-embedding-3-large
    deployment_name: text-embedding
    # encoding_model: cl100k_base # automatically set by tiktoken if left undefined
    model_supports_json: true # recommended if this is available for your model.
    concurrent_requests: 25 # max number of simultaneous LLM requests allowed
    async_mode: threaded # or asyncio
    retry_strategy: native
    max_retries: 10
    tokens_per_minute: auto              # set to null to disable rate limiting
    requests_per_minute: auto            # set to null to disable rate limiting
    dimensions: 3072

### Input settings ###

input:
  storage:
    type: blob # or blob
    base_dir: "input"
    connection_string: ""
    container_name: "graphrag"
    storage_account_blob_url: "https://stREPLACE_ME.blob.core.windows.net" # optional, used for blob storage
  file_type: text # [csv, text, json, pdf]
  

chunks:
  size: 1200
  overlap: 100
  group_by_columns: [id]

### Output/storage settings ###
## If blob storage is specified in the following four sections,
## connection_string and container_name must be provided

output:
  type: blob # [file, blob, cosmosdb]
  base_dir: "output"
  connection_string: ""
  container_name: "graphrag"
  storage_account_blob_url: "https://stREPLACE_ME.blob.core.windows.net" # optional, used for blob storage
    
cache:
  type: blob # [file, blob, cosmosdb]
  base_dir: "cache"
  connection_string: ""
  container_name: "graphrag"
  storage_account_blob_url: "https://stREPLACE_ME.blob.core.windows.net" # optional, used for blob storage

reporting:
  type: blob # [file, blob, cosmosdb]
  base_dir: "logs"
  connection_string: ""
  container_name: "graphrag"
  storage_account_blob_url: "https://stREPLACE_ME.blob.core.windows.net" # optional, used for blob storage

vector_store:
  default_vector_store:
    type: cosmosdb
    connection_string: "REPLACE_ME"
    database_name: vectorstore
    #db_uri: output\lancedb
    url: "https://cosmos-REPLACE_ME.documents.azure.com"
    container_name: graphrag
    overwrite: True
    settings:
      dimensions: 3072
  postgres_vector_store:
    type: postgresql
    host: psql-REPLACE_ME.postgres.database.azure.com
    port: 5432
    username: givenscj
    password: REPLACE_ME
    sslmode: require
    app_identity_name: your_app_identity_name # if using managed identity auth
    connection_string: "REPLACE_ME"
    database: vectorstore
    #db_uri: output\lancedb
    container_name: graphrag
    overwrite: True
    properties:
      azure_ml_scoring_endpoint: https://aif-REPLACE_ME.openai.azure.com/
      azure_ml_endpoint_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}
      azure_ml_deployment: ${AI_FOUNDRY_ACCOUNT_APIKEY}
      azure_openai_endpoint: "https://aif-REPLACE_ME.openai.azure.com/"
      azure_openai_endpoint_key: ${AI_FOUNDRY_ACCOUNT_APIKEY}
      azure_openai_deployment: "chat"
      index_type: diskann # [hnsw, diskann]
      run_post_embedding: true
      dimensions: 3072

### Workflow settings ###

embed_text:
  model_id: default_embedding_model
  vector_store_id: postgres_vector_store

extract_graph:
  model_id: default_chat_model
  prompt: "prompts/extract_graph.txt"
  entity_types: [organization,person,geo,event]
  max_gleanings: 1

summarize_descriptions:
  model_id: default_chat_model
  prompt: "prompts/summarize_descriptions.txt"
  max_length: 500

extract_graph_nlp:
  text_analyzer:
    extractor_type: regex_english # [regex_english, syntactic_parser, cfg]

cluster_graph:
  max_cluster_size: 10

extract_claims:
  enabled: false
  model_id: default_chat_model
  prompt: "prompts/extract_claims.txt"
  description: "Any claims or facts that could be relevant to information discovery."
  max_gleanings: 1

community_reports:
  model_id: default_chat_model
  graph_prompt: "prompts/community_report_graph.txt"
  text_prompt: "prompts/community_report_text.txt"
  max_length: 2000
  max_input_length: 8000

embed_graph:
  enabled: false # if true, will generate node2vec embeddings for nodes

umap:
  enabled: false # if true, will generate UMAP embeddings for nodes (embed_graph must also be enabled)

snapshots:
  graphml: false
  embeddings: false

### Query settings ###
## The prompt locations are required here, but each search method has a number of optional knobs that can be tuned.
## See the config docs: https://microsoft.github.io/graphrag/config/yaml/#query

local_search:
  chat_model_id: default_chat_model
  embedding_model_id: default_embedding_model
  prompt: "prompts/local_search_system_prompt.txt"

global_search:
  chat_model_id: default_chat_model
  map_prompt: "prompts/global_search_map_system_prompt.txt"
  reduce_prompt: "prompts/global_search_reduce_system_prompt.txt"
  knowledge_prompt: "prompts/global_search_knowledge_system_prompt.txt"

drift_search:
  chat_model_id: default_chat_model
  embedding_model_id: default_embedding_model
  prompt: "prompts/drift_search_system_prompt.txt"
  reduce_prompt: "prompts/drift_search_reduce_prompt.txt"

basic_search:
  chat_model_id: default_chat_model
  embedding_model_id: default_embedding_model
  prompt: "prompts/basic_search_system_prompt.txt"

"""

await save_prompt("gr_config", gr_config, container="docproc")